In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import random
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import seaborn as sns
from datetime import datetime
#import cv2

import re
import kaplanmeier as km

In [ ]:
from lifelines import CoxPHFitter

In [ ]:
#metablomics_data = pd.read_csv("./data/bruna/data_PA/metabolomics_raw.tsv", delimiter='\t')
#hematocrit_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/hemocrit_only_trimmed.txt", sep='\t')
#abdonminal_phenotypes_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/abdominal_merged.no_outliers.residuals.qnorm.txt", sep='\t')
cardiac_phenotype_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/cardiac_merged2.no_outliers.residuals.qnorm.txt" , sep='\t')
#protien_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/olink2_data_renamed.no_outliers.residuals.qnorm.txt", sep='\t')
#physical_activity_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/PA_merged.no_outliers.residuals.qnorm.txt", sep='\t')

big_pheno_data = pd.read_csv("./data/bruna/lvedv/ukbb_all_no_exclusion_all_2_and_3_with_CMR_and_MR", sep='\t', header=0)

unet_t1_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_latent_dimentions_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')
latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')
#latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/latent_dimensions_16_allpatients_PheWAS.no_outliers.medicalresiduals.qnorm.txt", sep='\t')

In [ ]:
#metablomics_data.replace(-1000, np.nan, inplace=True)
#abdonminal_phenotypes_data.replace(-1000, np.nan, inplace=True)
#hematocrit_data.replace(-1000, np.nan, inplace=True)
#protien_data.replace(-1000, np.nan, inplace=True)
#physical_activity_data.replace(-1000, np.nan, inplace=True)

big_pheno_data.replace(-1000, np.nan, inplace=True)

cardiac_phenotype_data.replace(-1000, np.nan, inplace=True)
unet_t1_regressed.replace(-1000, np.nan, inplace=True)
latent_dimensions.replace(-1000, np.nan, inplace=True)

#all_metablomics_data = pd.merge(metablomics_data, hematocrit_data, left_on="FID", right_on="FID")
#all_metablomics_data = all_metablomics_data.drop(columns = ["IID_x", "IID_y"])

In [ ]:
merged_data = pd.merge(unet_t1_regressed, big_pheno_data, left_on = "FID", right_on = "id")
survival_df = pd.DataFrame(merged_data['id'])
survival_data = pd.merge(survival_df, unet_t1_regressed, left_on="id", right_on="IID")
survival_data = pd.merge(survival_data, latent_dimensions, left_on="id", right_on="IID")
survival_data = survival_data.drop(["IID_x", "FID_x", "IID_y", "FID_y"], axis=1)

cardiac_phenotype_data['id'] = [str(int(float(ID))) for ID in cardiac_phenotype_data['id']]
survival_data['id'] = [str(int(float(ID))) for ID in survival_data['id']]

time to event = date of mri - 

{date of death
if else: date of lost follow-up
else: date of today}

In [ ]:
def calculate_age(year_of_birth, month_of_birth, date_death = None, study_end_date='2022-10-18'):
    # Convert month name to number (1-12)
    month_dict = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }

    if pd.isna(date_death):
        reference_date = datetime.strptime(study_end_date, '%Y-%m-%d')
        reference_year = reference_date.year
        reference_month = reference_date.month
    else:
        # Patient died, use date of death
        try:
            reference_date = datetime.strptime(date_death, '%Y-%m-%d')
            reference_year = reference_date.year
            reference_month = reference_date.month
        except:
            # Handle invalid date format
            return None
    
    # Calculate basic age
    age = reference_year - year_of_birth
    
    # Adjust if birthday hasn't occurred yet this year
    if reference_month < month_dict[month_of_birth]:
        age -= 1
        
    return age

calculate_age(1948.0, "February")

In [ ]:
big_pheno_data['current_age'] = big_pheno_data.apply(
    lambda row: calculate_age(row['year_of_birth'], row['month_of_birth'], row['date_death']),
    axis=1
)
merged_data = pd.merge(unet_t1_regressed, big_pheno_data, left_on = "FID", right_on = "id")

In [ ]:
# Create a histogram
plt.hist(merged_data['current_age'], bins=30, color='skyblue', edgecolor='black')

# Add labels and title
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title(f'Histogram of Age')

# Show the plot
plt.show()

In [ ]:
def estimate_mri_date(year_of_birth, month_of_birth, age_at_mri):
    """Estimate MRI date from birth info and age at MRI"""
    month_dict = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    
    mri_year = int(year_of_birth + age_at_mri)
    mri_month = month_dict[month_of_birth]
    
    # Use mid-month (15th) as approximation
    return pd.Timestamp(year=mri_year, month=mri_month, day=15)

# Apply to all rows
merged_data['estimated_mri_date'] = merged_data.apply(
    lambda row: estimate_mri_date(row['year_of_birth'], 
                                   row['month_of_birth'], 
                                   row['age_at_mri']), 
    axis=1
)

# Calculate time using DATES not ages
study_end = pd.to_datetime('2022-10-18')
merged_data['date_death_dt'] = pd.to_datetime(merged_data['date_death'])  # or date_death?

In [ ]:
# Prepare survival dataframe
survival_df = pd.DataFrame(merged_data['id'])

# Calculate time to event using dates (in years)
study_end = pd.to_datetime('2022-10-18')
merged_data['date_death_dt'] = pd.to_datetime(merged_data['date_death'])  # or 'date_death' if that's the right column

survival_df['time'] = (
    merged_data['date_death_dt'].fillna(study_end) - 
    merged_data['estimated_mri_date']
).dt.days / 365.25

# Event indicator (1 if died, 0 if alive/censored)
survival_df['event'] = merged_data['death_boolean'].astype(int)  # or use (~pd.isna(merged_data['date_death'])).astype(int)

# Group variable
survival_df['group'] = big_pheno_data['sex']

# Merge with other data
survival_data = pd.merge(survival_df, unet_t1_regressed, left_on="id", right_on="IID")
survival_data = pd.merge(survival_data, latent_dimensions, left_on="id", right_on="IID")
survival_data = survival_data.drop(["IID_x", "FID_x", "IID_y", "FID_y"], axis=1)

# Time to event in years
survival_df['time'] = (
    merged_data['date_death_dt'].fillna(study_end) - 
    merged_data['estimated_mri_date']
).dt.days / 365.25

survival_df['event'] = merged_data['death_boolean'].astype(int)

# Check results
print(survival_df['time'].describe())
deceased = survival_df[survival_df['event'] == 1]
for year in range(1, 12):
    print(f"< {year} years: {(deceased['time'] < year).sum()} deaths")

In [ ]:
import kaplanmeier as km
from lifelines import CoxPHFitter
import pandas as pd
import numpy as np
from scipy import stats

cph = CoxPHFitter()
survival_results = []

# Define cutoffs to test
cutoff_types = {
    'Median split': lambda x: x.median(),
    '25th percentile cutoff': lambda x: x.quantile(0.25),
    '75th percentile cutoff': lambda x: x.quantile(0.75),
    '2.5th percentile cutoff': lambda x: x.quantile(0.025),
    '97.5th percentile cutoff': lambda x: x.quantile(0.975)
}

for i in survival_data.columns:
    if i == "id" or i == "time" or i == "event" or i == "group":
        continue
    phenotype = i
    
    for cutoff_name, cutoff_func in cutoff_types.items():
        survival_data_copy = survival_data.copy()
        cutoff = cutoff_func(survival_data_copy[phenotype])
        
        # Create labeled groups
        survival_data_copy['group'] = np.where(
            survival_data_copy[phenotype] > cutoff, 
            'High', 
            'Low'
        )
        survival_data_copy = survival_data_copy.dropna(subset=['group'])
        
        # Skip if groups are too imbalanced
        if (survival_data_copy['group'] == 'High').sum() < 10 or (survival_data_copy['group'] == 'Low').sum() < 10:
            continue
        
        print(f"Analyzing: {phenotype} - {cutoff_name}")
        
        # Kaplan-Meier
        results = km.fit(survival_data_copy['time'], survival_data_copy['event'], survival_data_copy['group'])
        
        # Cox Proportional Hazards
        survival_data_copy['group_numeric'] = (survival_data_copy['group'] == 'High').astype(int)
        cox_data = survival_data_copy[['time', 'event', 'group_numeric']].copy()
        
        try:
            cph.fit(cox_data, duration_col='time', event_col='event')
            
            hr = cph.hazard_ratios_['group_numeric']
            ci = cph.confidence_intervals_.loc['group_numeric']
            ci_lower = np.exp(ci.iloc[0])
            ci_upper = np.exp(ci.iloc[1])
            cox_p_value = cph.summary.loc['group_numeric', 'p']
            c_index = cph.concordance_index_  # ADD THIS
            
            survival_results.append({
                'Phenotype': phenotype,
                'Stratification': cutoff_name,
                'Cutoff_Value': cutoff,
                'Logrank_P': results['logrank_P'],
                'Logrank_Z': results['logrank_Z'],
                'Hazard_Ratio': hr,
                'HR_95CI_Lower': ci_lower,
                'HR_95CI_Upper': ci_upper,
                'Cox_P': cox_p_value,
                'C_index': c_index,  # ADD THIS
                'N_High': (survival_data_copy['group'] == 'High').sum(),
                'N_Low': (survival_data_copy['group'] == 'Low').sum()
            })
            
        except Exception as e:
            print(f"  Cox regression failed: {e}")

# Create and save DataFrame
survival_results_df = pd.DataFrame(survival_results)
survival_results_df = survival_results_df.sort_values('Logrank_P')
print("\n" + "="*100)
print("="*100)
print(survival_results_df.head(20).to_string(index=False))

In [ ]:
survival_results_df
survival_results_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/survival_analysis_results.csv', index=False)

In [ ]:
from lifelines.statistics import multivariate_logrank_test

print("\n" + "="*100)
print("NESTED MODEL COMPARISON: Mean T1 vs Mean T1 + VAE Latent Dimensions")
print("="*100)

# Identify your mean T1 column name and VAE latent dimension columns
mean_t1_col = 'Mean_T1'  # ADJUST THIS to your actual column name
vae_cols = [col for col in survival_data.columns if 'latent' in col.lower() or 'LD' in col]  # ADJUST THIS

# Prepare data for multivariate Cox regression
multivar_data = survival_data[['time', 'event', mean_t1_col] + vae_cols].dropna()

# Model 1: Mean T1 only (baseline)
cph_baseline = CoxPHFitter()
cph_baseline.fit(multivar_data[['time', 'event', mean_t1_col]], 
                 duration_col='time', event_col='event')

# Model 2: Mean T1 + VAE latent dimensions
cph_full = CoxPHFitter()
cph_full.fit(multivar_data[['time', 'event', mean_t1_col] + vae_cols], 
             duration_col='time', event_col='event')

# Calculate likelihood ratio test
lr_stat = 2 * (cph_full.log_likelihood_ - cph_baseline.log_likelihood_)
df_diff = len(vae_cols)  # Difference in number of parameters
lr_p_value = stats.chi2.sf(lr_stat, df_diff)

# Print results
print(f"\nBaseline Model (Mean T1 only):")
print(f"  Log-likelihood: {cph_baseline.log_likelihood_:.4f}")
print(f"  C-index: {cph_baseline.concordance_index_:.4f}")
print(f"  AIC: {cph_baseline.AIC_partial_:.4f}")

print(f"\nFull Model (Mean T1 + {len(vae_cols)} VAE latent dimensions):")
print(f"  Log-likelihood: {cph_full.log_likelihood_:.4f}")
print(f"  C-index: {cph_full.concordance_index_:.4f}")
print(f"  AIC: {cph_full.AIC_partial_:.4f}")

print(f"\nLikelihood Ratio Test:")
print(f"  LR statistic: {lr_stat:.4f}")
print(f"  Degrees of freedom: {df_diff}")
print(f"  P-value: {lr_p_value:.2e}")
print(f"  C-index improvement: {cph_full.concordance_index_ - cph_baseline.concordance_index_:.4f}")

if lr_p_value < 0.05:
    print(f"  ✓ Adding VAE latent dimensions significantly improves model fit (p < 0.05)")
else:
    print(f"  ✗ Adding VAE latent dimensions does not significantly improve model fit (p ≥ 0.05)")

# Optional: Show individual VAE dimension contributions in the full model
print(f"\nVAE Latent Dimensions in Full Model:")
print(cph_full.summary[['coef', 'exp(coef)', 'p']].loc[vae_cols].sort_values('p'))

In [ ]:
# ========== INDIVIDUAL NESTED MODEL COMPARISONS ==========
print("\n" + "="*100)
print("INDIVIDUAL NESTED MODEL COMPARISONS: Mean T1 vs Mean T1 + Each VAE Dimension")
print("="*100)

# Identify column names
mean_t1_col = 'Mean_T1'  # ADJUST THIS to match your actual column name
vae_cols = [col for col in survival_data.columns if 'latent' in col.lower() or 'LD' in col]

# Store results
individual_results = []

# Fit baseline model once
multivar_data = survival_data[['time', 'event', mean_t1_col] + vae_cols].dropna()
cph_baseline = CoxPHFitter()
cph_baseline.fit(multivar_data[['time', 'event', mean_t1_col]], 
                 duration_col='time', event_col='event')

print(f"\nBaseline Model (Mean T1 only):")
print(f"  C-index: {cph_baseline.concordance_index_:.4f}")
print(f"  Log-likelihood: {cph_baseline.log_likelihood_:.4f}")
print(f"  AIC (partial): {cph_baseline.AIC_partial_:.4f}")

print(f"\nTesting addition of each VAE dimension individually:")
print("-" * 100)

# Test each VAE dimension individually
for vae_col in vae_cols:
    try:
        # Fit model with mean T1 + this specific VAE dimension
        cph_individual = CoxPHFitter()
        cph_individual.fit(multivar_data[['time', 'event', mean_t1_col, vae_col]], 
                          duration_col='time', event_col='event')
        
        # Likelihood ratio test
        lr_stat = 2 * (cph_individual.log_likelihood_ - cph_baseline.log_likelihood_)
        lr_p_value = stats.chi2.sf(lr_stat, 1)  # df=1 for one additional parameter
        
        # C-index improvement
        c_index_improvement = cph_individual.concordance_index_ - cph_baseline.concordance_index_
        
        # Get VAE coefficient info
        vae_hr = cph_individual.hazard_ratios_[vae_col]
        vae_p = cph_individual.summary.loc[vae_col, 'p']
        
        individual_results.append({
            'VAE_Dimension': vae_col,
            'C_index': cph_individual.concordance_index_,
            'C_index_improvement': c_index_improvement,
            'LR_statistic': lr_stat,
            'LR_p_value': lr_p_value,
            'VAE_HR': vae_hr,
            'VAE_coef_p': vae_p,
            'AIC_partial': cph_individual.AIC_partial_,
            'AIC_improvement': cph_baseline.AIC_partial_ - cph_individual.AIC_partial_
        })
        
    except Exception as e:
        print(f"Failed for {vae_col}: {e}")

# Create results dataframe and sort by C-index improvement
individual_df = pd.DataFrame(individual_results)
individual_df = individual_df.sort_values('C_index_improvement', ascending=False)

print("\nResults (sorted by C-index improvement):")
print(individual_df.to_string(index=False))

# Highlight significant improvements
print("\n" + "="*100)
print("SIGNIFICANT IMPROVEMENTS (LR test p < 0.05):")
print("="*100)
significant = individual_df[individual_df['LR_p_value'] < 0.05]
print(f"\n{len(significant)} out of {len(vae_cols)} VAE dimensions significantly improve model beyond mean T1 alone")
if len(significant) > 0:
    print(significant[['VAE_Dimension', 'C_index_improvement', 'LR_p_value', 'VAE_HR']].to_string(index=False))

# Summary statistics
print("\n" + "="*100)
print("SUMMARY:")
print("="*100)
print(f"Baseline (Mean T1 only) C-index: {cph_baseline.concordance_index_:.4f}")
if len(individual_df) > 0:
    print(f"Best individual VAE dimension C-index: {individual_df.iloc[0]['C_index']:.4f}")
    print(f"Best C-index improvement: {individual_df.iloc[0]['C_index_improvement']:.4f}")
    print(f"Mean C-index improvement across all VAE dimensions: {individual_df['C_index_improvement'].mean():.4f}")
    print(f"Number with positive C-index improvement: {(individual_df['C_index_improvement'] > 0).sum()}/{len(vae_cols)}")

In [ ]:
# ── Mortality Curves: Strain Phenotypes ──────────────────────────────────────

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Exact strain column names
strain_cols = [
    # Circumferential strain (AHA segments 1-16)
    'LV.circumferential.strain.AHA.1', 'LV.circumferential.strain.AHA.2',
    'LV.circumferential.strain.AHA.3', 'LV.circumferential.strain.AHA.4',
    'LV.circumferential.strain.AHA.5', 'LV.circumferential.strain.AHA.6',
    'LV.circumferential.strain.AHA.7', 'LV.circumferential.strain.AHA.8',
    'LV.circumferential.strain.AHA.9', 'LV.circumferential.strain.AHA.10',
    'LV.circumferential.strain.AHA.11', 'LV.circumferential.strain.AHA.12',
    'LV.circumferential.strain.AHA.13', 'LV.circumferential.strain.AHA.14',
    'LV.circumferential.strain.AHA.15', 'LV.circumferential.strain.AHA.16',
    # Longitudinal strain (6 segments)
    'LV.longitudinal.strain.Segment.1', 'LV.longitudinal.strain.Segment.2',
    'LV.longitudinal.strain.Segment.3', 'LV.longitudinal.strain.Segment.4',
    'LV.longitudinal.strain.Segment.5', 'LV.longitudinal.strain.Segment.6',
    # Radial strain (AHA segments 1-16)
    'LV.radial.strain.AHA.1',  'LV.radial.strain.AHA.2',
    'LV.radial.strain.AHA.3',  'LV.radial.strain.AHA.4',
    'LV.radial.strain.AHA.5',  'LV.radial.strain.AHA.6',
    'LV.radial.strain.AHA.7',  'LV.radial.strain.AHA.8',
    'LV.radial.strain.AHA.9',  'LV.radial.strain.AHA.10',
    'LV.radial.strain.AHA.11', 'LV.radial.strain.AHA.12',
    'LV.radial.strain.AHA.13', 'LV.radial.strain.AHA.14',
    'LV.radial.strain.AHA.15', 'LV.radial.strain.AHA.16',
]

cutoff_types = {
    'Median split':           lambda x: x.median(),
    '25th percentile cutoff': lambda x: x.quantile(0.25),
    '75th percentile cutoff': lambda x: x.quantile(0.75),
    '2.5th percentile cutoff':  lambda x: x.quantile(0.025),
    '97.5th percentile cutoff': lambda x: x.quantile(0.975),
}

# Merge cardiac phenotype (strain) data into survival_data on 'id'
survival_with_strain = pd.merge(
    survival_data,
    cardiac_phenotype_data[['id'] + strain_cols],
    on='id',
    how='inner'
)
survival_with_strain.replace(-1000, np.nan, inplace=True)

print(f"Patients after merge: {len(survival_with_strain)}  |  Deaths: {survival_with_strain['event'].sum()}")

strain_groups = {
    'Circumferential': [c for c in strain_cols if 'circumferential' in c],
    'Longitudinal':    [c for c in strain_cols if 'longitudinal'    in c],
    'Radial':          [c for c in strain_cols if 'radial'          in c],
}

summary_rows = []

for cutoff_name, cutoff_func in cutoff_types.items():
    print(f"\n{'='*90}")
    print(f"CUTOFF: {cutoff_name}")
    print(f"{'='*90}")

    for strain_type, cols in strain_groups.items():
        n_cols = len(cols)
        n_rows = (n_cols + 1) // 2
        fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 5))
        axes = axes.flatten()
        fig.suptitle(f'Mortality Curves — LV {strain_type} Strain\n{cutoff_name}',
                     fontsize=15, fontweight='bold', y=1.01)

        for ax_idx, phenotype in enumerate(cols):
            ax = axes[ax_idx]
            df = survival_with_strain[['time', 'event', phenotype]].dropna().copy()

            if len(df) < 20:
                ax.set_visible(False)
                continue

            cutoff = cutoff_func(df[phenotype])
            df['group'] = np.where(df[phenotype] > cutoff, 'High', 'Low')
            high = df[df['group'] == 'High']
            low  = df[df['group'] == 'Low']

            if len(high) < 10 or len(low) < 10:
                ax.set_visible(False)
                continue

            lr = logrank_test(low['time'], high['time'], low['event'], high['event'])
            p  = lr.p_value
            stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

            kmf = KaplanMeierFitter()
            for grp, label, color in [('Low',  f'Low  n={len(low)}',  '#2196F3'),
                                       ('High', f'High n={len(high)}', '#F44336')]:
                subset = df[df['group'] == grp]
                kmf.fit(subset['time'], subset['event'], label=label)
                kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

            segment = phenotype.split('.')[-1]
            ax.set_title(f'Segment {segment}  |  p={p:.2e}  {stars}', fontsize=10)
            ax.set_xlabel('Time from MRI (years)')
            ax.set_ylabel('Survival probability')
            ax.legend(fontsize=8, loc='lower left')
            ax.grid(alpha=0.3)

            summary_rows.append({
                'Phenotype':      phenotype,
                'Strain_Type':    strain_type,
                'Segment':        segment,
                'Cutoff_Name':    cutoff_name,
                'Cutoff_Value':   cutoff,
                'Logrank_P':      p,
                'Significant':    p < 0.05,
                'N_High':         len(high),
                'N_Low':          len(low),
                'N_Events_High':  int(high['event'].sum()),
                'N_Events_Low':   int(low['event'].sum()),
            })

        for ax_idx in range(len(cols), len(axes)):
            axes[ax_idx].set_visible(False)

        plt.tight_layout()
        plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows).sort_values(['Cutoff_Name', 'Logrank_P'])
print("\n" + "="*90)
print("STRAIN PHENOTYPE SURVIVAL ANALYSIS — SUMMARY")
print("="*90)
print(summary_df.to_string(index=False))

sig = summary_df[summary_df['Significant']]
print(f"\n{len(sig)} / {len(summary_df)} strain-cutoff combinations significantly associated with mortality (p < 0.05)")

summary_df.to_csv(
    './data/shriya/SHMOLLI-output-unet-myocardium-update2/strain_survival_results.csv',
    index=False
)

In [ ]:
summary_df[summary_df["Logrank_P"] < 0.0002631578947368421]


In [ ]:
# ── Hazard Ratios for Most Significant Strain Phenotypes ─────────────────────

from lifelines import CoxPHFitter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Define the significant combinations to test
significant_combinations = [
    ('LV.radial.strain.AHA.10',          '2.5th percentile cutoff'),
    ('LV.radial.strain.AHA.9',           '2.5th percentile cutoff'),
    ('LV.radial.strain.AHA.5',           '2.5th percentile cutoff'),
    ('LV.circumferential.strain.AHA.14', '97.5th percentile cutoff'),
    ('LV.circumferential.strain.AHA.15', '97.5th percentile cutoff'),
    ('LV.circumferential.strain.AHA.8',  '97.5th percentile cutoff'),
    ('LV.circumferential.strain.AHA.13', '97.5th percentile cutoff'),
    ('LV.circumferential.strain.AHA.4',  '97.5th percentile cutoff'),
]

cutoff_types = {
    'Median split':             lambda x: x.median(),
    '25th percentile cutoff':   lambda x: x.quantile(0.25),
    '75th percentile cutoff':   lambda x: x.quantile(0.75),
    '2.5th percentile cutoff':  lambda x: x.quantile(0.025),
    '97.5th percentile cutoff': lambda x: x.quantile(0.975),
}

cph = CoxPHFitter()
hr_results = []

for phenotype, cutoff_name in significant_combinations:
    df = survival_with_strain[['time', 'event', phenotype]].dropna().copy()

    cutoff = cutoff_types[cutoff_name](df[phenotype])
    df['group_numeric'] = (df[phenotype] > cutoff).astype(int)  # 1=High, 0=Low

    try:
        cph.fit(df[['time', 'event', 'group_numeric']], duration_col='time', event_col='event')

        hr        = cph.hazard_ratios_['group_numeric']
        ci        = cph.confidence_intervals_.loc['group_numeric']
        ci_lower  = np.exp(ci.iloc[0])
        ci_upper  = np.exp(ci.iloc[1])
        cox_p     = cph.summary.loc['group_numeric', 'p']
        c_index   = cph.concordance_index_

        hr_results.append({
            'Phenotype':    phenotype,
            'Cutoff_Name':  cutoff_name,
            'Cutoff_Value': cutoff,
            'Hazard_Ratio': hr,
            'HR_95CI_Lower': ci_lower,
            'HR_95CI_Upper': ci_upper,
            'Cox_P':        cox_p,
            'C_index':      c_index,
            'N_High':       int((df['group_numeric'] == 1).sum()),
            'N_Low':        int((df['group_numeric'] == 0).sum()),
        })

        print(f"{phenotype} | {cutoff_name}")
        print(f"  HR={hr:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f}) | Cox p={cox_p:.3e} | C-index={c_index:.3f}\n")

    except Exception as e:
        print(f"Cox failed for {phenotype} / {cutoff_name}: {e}")

hr_df = pd.DataFrame(hr_results).sort_values('Cox_P')

# ── Forest plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, len(hr_df) * 0.7 + 2))

colors = {'2.5th percentile cutoff': '#E53935', '97.5th percentile cutoff': '#1E88E5'}

for i, row in hr_df.reset_index(drop=True).iterrows():
    y       = len(hr_df) - i  # top to bottom
    color   = colors.get(row['Cutoff_Name'], '#333333')
    stars   = '***' if row['Cox_P'] < 0.001 else ('**' if row['Cox_P'] < 0.01 else '*')

    # CI line
    ax.plot([row['HR_95CI_Lower'], row['HR_95CI_Upper']], [y, y],
            color=color, linewidth=2, zorder=2)
    # HR point
    ax.scatter(row['Hazard_Ratio'], y, color=color, s=80, zorder=3)

    # Label: short phenotype name + stars
    short_name = row['Phenotype'].replace('LV.', '').replace('.strain', '').replace('.', ' ')
    ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else 0.01,
            y, f"{short_name}  {stars}",
            va='center', ha='right', fontsize=9, color=color)

ax.axvline(x=1, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=11)
ax.set_title('Cox Hazard Ratios — Most Significant Strain Phenotypes', fontsize=13, fontweight='bold')
ax.set_yticks([])
ax.set_ylim(0.5, len(hr_df) + 0.5)

legend_patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

# ── Table ─────────────────────────────────────────────────────────────────────
print("\n" + "="*90)
print("HAZARD RATIO SUMMARY")
print("="*90)
print(hr_df[['Phenotype','Cutoff_Name','Hazard_Ratio','HR_95CI_Lower',
             'HR_95CI_Upper','Cox_P','C_index']].to_string(index=False))


**Note on the next 5 cells (nested Cox model comparison):** these are kept as sequential methodological refinements rather than collapsed into one, since it wasn't possible to confirm from source alone which version's numbers are the ones reported in the manuscript. Each cell fixes an issue in the one before it: cell 1/5 tests combined 'clinical' feature bundles (T1+LVMI+LVEF) against strain/VAE additions with a small regularization penalty; cell 2/5 instead tests each covariate (strain, LVMI, LVEF, VAE) added individually to the T1-only base model, still penalized; cell 3/5 switches to an unpenalized fit because a penalized fit invalidates the likelihood-ratio test, and restricts every model to one shared complete-case sample so the LRTs are comparable -- note its final print statement claims results were saved but the save call is commented out; cell 4/5 is a 3-line diagnostic (events-per-variable) that shows the shared complete-case sample from cell 3/5 drops too many patients for some models; cell 5/5 responds by fitting each model on its own maximum available sample instead, with per-model sample-matched likelihood-ratio tests against a refit base model. Confirm against the manuscript's reported C-index/HR/p-values before treating any one of these as canonical.

In [ ]:
# ── Nested Cox Model Comparison ──────────────────────────────────────────────

from lifelines import CoxPHFitter
from scipy import stats
import pandas as pd
import numpy as np

# ── Define feature sets ───────────────────────────────────────────────────────

# Adjust these column names to match your actual data
AGE_COL   = 'current_age'
SEX_COL   = 'sex'
T1_COL    = 'Mean_T1'
LVMI_COL  = 'Left.ventricular.mass'          # adjust if needed
LVEF_COL  = 'LV.ejection.fraction'                            # adjust if needed
VAE_COLS = [col for col in survival_data.columns if 'Latent' in col or 'latent' in col]

GLS_COLS  = [f'LV.longitudinal.strain.Segment.{i}' for i in range(1, 7)]
GCS_COLS  = [f'LV.circumferential.strain.AHA.{i}'  for i in range(1, 17)]
STRAIN_COLS = GLS_COLS + GCS_COLS

VAE_COLS  = [col for col in survival_data.columns if 'Latent' in col or 'latent' in col]

# ── Build a merged dataframe with everything needed ───────────────────────────

cox_base = survival_data[['id', 'time', 'event']].copy()
cox_base['id'] = cox_base['id'].astype(int)
cardiac_phenotype_data['id'] = cardiac_phenotype_data['id'].apply(lambda x: int(float(x))).astype(int)

# Attach age + sex from big_pheno_data
cox_base = cox_base.merge(
    big_pheno_data[['id', AGE_COL, SEX_COL]].drop_duplicates('id'),
    on='id', how='inner'
)

# Attach Mean T1 (already in survival_data via unet_t1_regressed)
cox_base = cox_base.merge(
    survival_data[['id', T1_COL]],
    on='id', how='inner'
)

# Attach strain phenotypes
cox_base = cox_base.merge(
    cardiac_phenotype_data[['id'] + STRAIN_COLS],
    on='id', how='inner'
)

# Attach LVMI and LVEF from cardiac_phenotype_data
cox_base = cox_base.merge(
    cardiac_phenotype_data[['id', LVMI_COL, LVEF_COL]],
    on='id', how='inner'
)

cox_base = cox_base.merge(
    survival_data[['id'] + VAE_COLS],
    on='id', how='inner'
)

# Encode sex numerically if needed
if cox_base[SEX_COL].dtype == object:
    cox_base[SEX_COL] = (cox_base[SEX_COL] == cox_base[SEX_COL].unique()[0]).astype(int)

cox_base.replace(-1000, np.nan, inplace=True)
cox_base = cox_base[(cox_base['time'] > 0)].reset_index(drop=True)

print(f"Total patients available: {len(cox_base)}  |  Deaths: {cox_base['event'].sum()}")

# ── Define models ─────────────────────────────────────────────────────────────

BASE_FEATURES = [T1_COL]

models = {
    'X0 — Base (Mean T1)':                          BASE_FEATURES,
    'X1 — Clinical (Mean T1 + LVMI + LVEF)':        BASE_FEATURES + [LVMI_COL, LVEF_COL],
    'X2 — Clinical + Strain':                       BASE_FEATURES + [LVMI_COL, LVEF_COL] + STRAIN_COLS,
    'X3 — Clinical + VAE':                          BASE_FEATURES + [LVMI_COL, LVEF_COL] + VAE_COLS,
    'X4 — Clinical + Strain + VAE':                 BASE_FEATURES + [LVMI_COL, LVEF_COL] + STRAIN_COLS + VAE_COLS,
}

# ── Fit all models and collect metrics ───────────────────────────────────────

cph = CoxPHFitter(penalizer=0.1)   # small penalty for stability with many features
fitted_models = {}
model_results = []

for model_name, features in models.items():
    required_cols = ['time', 'event'] + features
    df_model = cox_base[required_cols].dropna()

    if len(df_model) < 50:
        print(f"Skipping {model_name}: too few complete cases ({len(df_model)})")
        continue

    try:
        cph.fit(df_model, duration_col='time', event_col='event')
        fitted_models[model_name] = (cph, df_model, features)

        model_results.append({
            'Model':        model_name,
            'N':            len(df_model),
            'N_Events':     int(df_model['event'].sum()),
            'C_index':      cph.concordance_index_,
            'Log_Likelihood': cph.log_likelihood_,
            'AIC':          cph.AIC_partial_,
            'N_Features':   len(features),
        })

        print(f"\n{model_name}")
        print(f"  N={len(df_model)} | Events={df_model['event'].sum()} | "
              f"C-index={cph.concordance_index_:.4f} | AIC={cph.AIC_partial_:.2f}")

    except Exception as e:
        print(f"  Failed: {e}")

results_df = pd.DataFrame(model_results)

# ── Likelihood ratio tests vs base model ─────────────────────────────────────

print("\n" + "="*90)
print("LIKELIHOOD RATIO TESTS vs X0 (Base Model)")
print("="*90)

base_name = 'X0 — Base (Mean T1)'
base_ll   = results_df.loc[results_df['Model'] == base_name, 'Log_Likelihood'].values[0]
base_ci   = results_df.loc[results_df['Model'] == base_name, 'C_index'].values[0]
base_n    = len(models[base_name])

lrt_rows = []
for _, row in results_df.iterrows():
    if row['Model'] == base_name:
        continue
    df_diff = row['N_Features'] - base_n
    if df_diff <= 0:
        continue
    lr_stat = 2 * (row['Log_Likelihood'] - base_ll)
    p_val   = stats.chi2.sf(lr_stat, df_diff)
    ci_imp  = row['C_index'] - base_ci
    stars   = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))

    lrt_rows.append({
        'Model':           row['Model'],
        'C_index':         row['C_index'],
        'C_index_vs_base': ci_imp,
        'LR_stat':         lr_stat,
        'df':              df_diff,
        'P_value':         p_val,
        'Sig':             stars,
        'AIC':             row['AIC'],
    })
    print(f"\n  {row['Model']}")
    print(f"    C-index={row['C_index']:.4f} (Δ={ci_imp:+.4f}) | "
          f"LR={lr_stat:.2f} df={df_diff} p={p_val:.3e} {stars} | AIC={row['AIC']:.2f}")

lrt_df = pd.DataFrame(lrt_rows)

# ── Forest / C-index comparison plot ─────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

model_labels = results_df['Model'].str.extract(r'^(X[\d\+]+)')[0]
colors_list  = ['#455A64'] + ['#E53935','#1E88E5','#43A047','#8E24AA','#FB8C00','#00ACC1']

bars = ax.barh(results_df['Model'], results_df['C_index'],
               color=colors_list[:len(results_df)], alpha=0.85, edgecolor='white')

# Annotate bars
for bar, (_, row) in zip(bars, results_df.iterrows()):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{row['C_index']:.4f}", va='center', fontsize=9)

ax.axvline(x=base_ci, color='#455A64', linestyle='--', linewidth=1.2, label='Base C-index')
ax.set_xlabel('C-index (concordance)', fontsize=11)
ax.set_title('Nested Cox Model Comparison\nC-index by Feature Set', fontsize=13, fontweight='bold')
ax.set_xlim(0.5, min(results_df['C_index'].max() + 0.05, 1.0))
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────

print("\n" + "="*90)
print("FULL MODEL COMPARISON TABLE")
print("="*90)
print(results_df[['Model','N','N_Events','C_index','AIC','N_Features']].to_string(index=False))

# Save
results_df.to_csv(
    './data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_comparison.csv',
    index=False
)
lrt_df.to_csv(
    './data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_lrt.csv',
    index=False
)

In [ ]:
# ── Nested Cox Model Comparison ──────────────────────────────────────────────

from lifelines import CoxPHFitter
from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Column definitions ────────────────────────────────────────────────────────

T1_COL    = 'Mean_T1'
AGE_COL   = 'current_age'
SEX_COL   = 'sex'
LVMI_COL  = 'Left.ventricular.mass'
LVEF_COL  = 'LV.ejection.fraction'

GLS_COLS    = [f'LV.longitudinal.strain.Segment.{i}' for i in range(1, 7)]
GCS_COLS    = [f'LV.circumferential.strain.AHA.{i}'  for i in range(1, 17)]
STRAIN_COLS = GLS_COLS + GCS_COLS

VAE_COLS = [col for col in survival_data.columns if 'Latent' in col or 'latent' in col]
print(f"VAE cols ({len(VAE_COLS)}): {VAE_COLS}")

# ── Build merged dataframe ────────────────────────────────────────────────────

# Start with survival data (has time, event, id, VAE, T1)
cox_base = survival_data[['id', 'time', 'event', T1_COL] + VAE_COLS].copy()

# Attach age + sex from big_pheno_data
cox_base = cox_base.merge(
    big_pheno_data[['id', AGE_COL, SEX_COL]].drop_duplicates('id'),
    on='id', how='inner'
)

# Encode sex as numeric if needed
if cox_base[SEX_COL].dtype == object:
    cox_base[SEX_COL] = (cox_base[SEX_COL] == cox_base[SEX_COL].unique()[0]).astype(int)

# Attach LVMI, LVEF, strain from cardiac_phenotype_data
cardiac_phenotype_data['id'] = cardiac_phenotype_data['id'].apply(lambda x: int(float(x)))
cox_base = cox_base.merge(
    cardiac_phenotype_data[['id', LVMI_COL, LVEF_COL] + STRAIN_COLS],
    on='id', how='inner'
)

cox_base.replace(-1000, np.nan, inplace=True)
cox_base = cox_base[cox_base['time'] > 0].reset_index(drop=True)

print(f"Total patients: {len(cox_base)}  |  Deaths: {int(cox_base['event'].sum())}\n")

# ── Model definitions ─────────────────────────────────────────────────────────

BASE = [T1_COL]

models = {
    'X0 — Base (T1)':             BASE,
    'X1 — Base + Strain':                     BASE + STRAIN_COLS,
    'X2 — Base + LVMI':                       BASE + [LVMI_COL],
    'X3 — Base + LVEF':                       BASE + [LVEF_COL],
    'X4 — Base + VAE':                        BASE + VAE_COLS,
}

model_colors = {
    'X0 — Base (T1)':             '#455A64',
    'X1 — Base + Strain':                     '#1E88E5',
    'X2 — Base + LVMI':                       '#E53935',
    'X3 — Base + LVEF':                       '#FB8C00',
    'X4 — Base + VAE':                        '#8E24AA',
}

# ── Fit models ────────────────────────────────────────────────────────────────

fitted_models = {}
model_summary = []

for model_name, features in models.items():
    df_model = cox_base[['time', 'event'] + features].dropna()

    if len(df_model) < 50:
        print(f"Skipping {model_name}: too few complete cases ({len(df_model)})")
        continue

    try:
        cph = CoxPHFitter(penalizer=0.1)
        cph.fit(df_model, duration_col='time', event_col='event')
        fitted_models[model_name] = (cph, df_model, features)

        model_summary.append({
            'Model':          model_name,
            'N':              len(df_model),
            'N_Events':       int(df_model['event'].sum()),
            'C_index':        round(cph.concordance_index_, 4),
            'Log_Likelihood': round(cph.log_likelihood_, 4),
            'AIC':            round(cph.AIC_partial_, 4),
            'N_Features':     len(features),
        })

        print(f"{model_name}")
        print(f"  N={len(df_model)} | Events={int(df_model['event'].sum())} | "
              f"C-index={cph.concordance_index_:.4f} | AIC={cph.AIC_partial_:.2f}\n")

    except Exception as e:
        print(f"  FAILED — {model_name}: {e}\n")

results_df = pd.DataFrame(model_summary)

# ── Likelihood ratio tests vs X0 ─────────────────────────────────────────────

print("=" * 80)
print("LIKELIHOOD RATIO TESTS vs X0 (Base)")
print("=" * 80)

base_row = results_df[results_df['Model'] == 'X0 — Base (T1)'].iloc[0]
base_ll  = base_row['Log_Likelihood']
base_ci  = base_row['C_index']
base_nf  = base_row['N_Features']

lrt_rows = []
for _, row in results_df.iterrows():
    if row['Model'] == 'X0 — Base (T1)':
        continue
    df_diff = int(row['N_Features'] - base_nf)
    lr_stat = 2 * (row['Log_Likelihood'] - base_ll)
    p_val   = stats.chi2.sf(lr_stat, df_diff)
    ci_imp  = round(row['C_index'] - base_ci, 4)
    stars   = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))

    lrt_rows.append({
        'Model':         row['Model'],
        'C_index':       row['C_index'],
        'Delta_C_index': ci_imp,
        'LR_stat':       round(lr_stat, 3),
        'df':            df_diff,
        'P_value':       p_val,
        'Sig':           stars,
    })

    print(f"  {row['Model']}")
    print(f"    C-index={row['C_index']:.4f} (Δ={ci_imp:+.4f}) | "
          f"LR={lr_stat:.2f} df={df_diff} p={p_val:.3e} {stars}\n")

lrt_df = pd.DataFrame(lrt_rows)

# ── Plot 1: C-index bar chart ─────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
colors_list = [model_colors[m] for m in results_df['Model']]
bars = ax.barh(results_df['Model'], results_df['C_index'],
               color=colors_list, alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars, results_df.iterrows()):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{row['C_index']:.4f}", va='center', fontsize=9)

ax.axvline(x=base_ci, color='#455A64', linestyle='--', linewidth=1.2, label='Base C-index')
ax.set_xlabel('C-index (concordance)', fontsize=11)
ax.set_title('Nested Cox Model Comparison — C-index by Feature Set',
             fontsize=13, fontweight='bold')
ax.set_xlim(0.5, min(results_df['C_index'].max() + 0.05, 1.0))
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Plot 2: Forest plot per model (only that model's features) ────────────────

for model_name, (cph, df_model, features) in fitted_models.items():
    s = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
    s.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
    s['stars'] = s['p_value'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
    s = s.sort_values('HR', ascending=True)
    color = model_colors[model_name]
    c_idx = results_df.loc[results_df['Model'] == model_name, 'C_index'].values[0]

    fig, ax = plt.subplots(figsize=(10, max(4, len(s) * 0.4 + 2)))
    for i, (feat, row) in enumerate(s.iterrows()):
        ax.plot([row['CI_lower'], row['CI_upper']], [i, i],
                color=color, linewidth=2, alpha=0.8)
        ax.scatter(row['HR'], i, color=color, s=60, zorder=3)
        ax.text(-0.01, i, f"{feat}  {row['stars']}",
                va='center', ha='right', fontsize=8,
                transform=ax.get_yaxis_transform())

    ax.axvline(x=1, color='grey', linestyle='--', linewidth=1)
    ax.set_yticks([])
    ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=11)
    ax.set_title(f'Forest Plot — {model_name}  |  C-index={c_idx:.4f}',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.subplots_adjust(left=0.35)
    plt.tight_layout()
    plt.show()

# ── Save ──────────────────────────────────────────────────────────────────────

# Full HR table
all_hr_rows = []
for model_name, (cph, df_model, features) in fitted_models.items():
    s = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
    s.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
    s['Model']   = model_name
    s['Feature'] = s.index
    all_hr_rows.append(s)
all_hr_df = pd.concat(all_hr_rows).reset_index(drop=True)

results_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_comparison.csv', index=False)
lrt_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_lrt.csv', index=False)
all_hr_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_hazard_ratios.csv', index=False)

print("Done. All results saved.")

In [ ]:
# ── Nested Cox Model Comparison ──────────────────────────────────────────────
# NOTE: All models fit UNPENALIZED for valid LRT hypothesis testing.
# C-indices here are from unpenalized fits — interpret with that in mind.

from lifelines import CoxPHFitter
from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Column definitions ────────────────────────────────────────────────────────

T1_COL    = 'Mean_T1'
AGE_COL   = 'current_age'
SEX_COL   = 'sex'
LVMI_COL  = 'Left.ventricular.mass'
LVEF_COL  = 'LV.ejection.fraction'

GLS_COLS    = [f'LV.longitudinal.strain.Segment.{i}' for i in range(1, 7)]
GCS_COLS    = [f'LV.circumferential.strain.AHA.{i}'  for i in range(1, 17)]
STRAIN_COLS = GLS_COLS + GCS_COLS

VAE_COLS = [col for col in survival_data.columns if 'Latent' in col or 'latent' in col]
print(f"VAE cols ({len(VAE_COLS)}): {VAE_COLS}")

BASE = [T1_COL]

# ── Build merged dataframe ────────────────────────────────────────────────────

cox_base = survival_data[['id', 'time', 'event', T1_COL] + VAE_COLS].copy()

cox_base = cox_base.merge(
    big_pheno_data[['id', AGE_COL, SEX_COL]].drop_duplicates('id'),
    on='id', how='inner'
)

if cox_base[SEX_COL].dtype == object:
    cox_base[SEX_COL] = (cox_base[SEX_COL] == cox_base[SEX_COL].unique()[0]).astype(int)

cardiac_phenotype_data['id'] = cardiac_phenotype_data['id'].apply(lambda x: int(float(x)))
cox_base = cox_base.merge(
    cardiac_phenotype_data[['id', LVMI_COL, LVEF_COL] + STRAIN_COLS],
    on='id', how='inner'
)

cox_base.replace(-1000, np.nan, inplace=True)
cox_base = cox_base[cox_base['time'] > 0].reset_index(drop=True)

print(f"Total patients: {len(cox_base)}  |  Deaths: {int(cox_base['event'].sum())}\n")

# ── Use the same complete-case sample for all models ─────────────────────────
# Drop rows missing ANY column used across all models so LRTs are comparable

all_features = BASE + STRAIN_COLS + [LVMI_COL, LVEF_COL] + VAE_COLS
cox_complete = cox_base[['time', 'event'] + all_features].dropna().reset_index(drop=True)

print(f"Complete cases (all models): {len(cox_complete)}  |  Deaths: {int(cox_complete['event'].sum())}\n")

# ── Model definitions ─────────────────────────────────────────────────────────

models = {
    'X0 — Base (T1 + age + sex)': BASE,
    'X1 — Base + Strain':         BASE + STRAIN_COLS,
    'X2 — Base + LVMI':           BASE + [LVMI_COL],
    'X3 — Base + LVEF':           BASE + [LVEF_COL],
    'X4 — Base + VAE':            BASE + VAE_COLS,
}

model_colors = {
    'X0 — Base (T1 + age + sex)': '#455A64',
    'X1 — Base + Strain':         '#1E88E5',
    'X2 — Base + LVMI':           '#E53935',
    'X3 — Base + LVEF':           '#FB8C00',
    'X4 — Base + VAE':            '#8E24AA',
}

# ── Fit all models unpenalized on the same sample ─────────────────────────────

fitted_models = {}
model_summary = []

for model_name, features in models.items():
    df_model = cox_complete[['time', 'event'] + features]

    try:
        cph = CoxPHFitter(penalizer=0.0)   # unpenalized for valid LRT
        cph.fit(df_model, duration_col='time', event_col='event')
        fitted_models[model_name] = (cph, df_model, features)

        model_summary.append({
            'Model':          model_name,
            'N':              len(df_model),
            'N_Events':       int(df_model['event'].sum()),
            'C_index':        round(cph.concordance_index_, 4),
            'Log_Likelihood': round(cph.log_likelihood_, 4),
            'AIC':            round(cph.AIC_partial_, 4),
            'N_Features':     len(features),
        })

        print(f"{model_name}")
        print(f"  N={len(df_model)} | Events={int(df_model['event'].sum())} | "
              f"C-index={cph.concordance_index_:.4f} | "
              f"LL={cph.log_likelihood_:.4f} | AIC={cph.AIC_partial_:.2f}\n")

    except Exception as e:
        print(f"  FAILED — {model_name}: {e}\n")

results_df = pd.DataFrame(model_summary)

# ── Likelihood ratio tests vs X0 ─────────────────────────────────────────────

print("=" * 80)
print("LIKELIHOOD RATIO TESTS vs X0 (unpenalized, same sample)")
print("=" * 80)

base_row = results_df[results_df['Model'] == 'X0 — Base (T1 + age + sex)'].iloc[0]
base_ll  = base_row['Log_Likelihood']
base_ci  = base_row['C_index']
base_nf  = base_row['N_Features']

lrt_rows = []
for _, row in results_df.iterrows():
    if row['Model'] == 'X0 — Base (T1 + age + sex)':
        continue

    df_diff = int(row['N_Features'] - base_nf)
    lr_stat = 2 * (row['Log_Likelihood'] - base_ll)
    p_val   = stats.chi2.sf(lr_stat, df_diff)
    ci_imp  = round(row['C_index'] - base_ci, 4)
    stars   = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))

    lrt_rows.append({
        'Model':         row['Model'],
        'C_index':       row['C_index'],
        'Delta_C_index': ci_imp,
        'LR_stat':       round(lr_stat, 3),
        'df':            df_diff,
        'P_value':       p_val,
        'Sig':           stars,
        'AIC':           row['AIC'],
    })

    print(f"  {row['Model']}")
    print(f"    C-index={row['C_index']:.4f} (Δ={ci_imp:+.4f}) | "
          f"LR={lr_stat:.2f} df={df_diff} p={p_val:.3e} {stars}\n")

lrt_df = pd.DataFrame(lrt_rows)

# ── Plot 1: C-index bar chart ─────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
colors_list = [model_colors[m] for m in results_df['Model']]
bars = ax.barh(results_df['Model'], results_df['C_index'],
               color=colors_list, alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars, results_df.iterrows()):
    lrt_row = lrt_df[lrt_df['Model'] == row['Model']]
    sig_label = f"  {lrt_row['Sig'].values[0]}" if len(lrt_row) > 0 else ''
    ax.text(bar.get_width() + 0.001,
            bar.get_y() + bar.get_height() / 2,
            f"{row['C_index']:.4f}{sig_label}",
            va='center', fontsize=9)

ax.axvline(x=base_ci, color='#455A64', linestyle='--', linewidth=1.2, label='Base C-index')
ax.set_xlabel('C-index (concordance)', fontsize=11)
ax.set_title('Nested Cox Model Comparison — C-index by Feature Set\n(unpenalized, LRT significance shown)',
             fontsize=12, fontweight='bold')
ax.set_xlim(0.5, min(results_df['C_index'].max() + 0.07, 1.0))
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Plot 2: Forest plot per model ─────────────────────────────────────────────

for model_name, (cph, df_model, features) in fitted_models.items():
    s = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
    s.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
    s['stars'] = s['p_value'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
    s = s.sort_values('HR', ascending=True)
    color = model_colors[model_name]
    c_idx = results_df.loc[results_df['Model'] == model_name, 'C_index'].values[0]

    fig, ax = plt.subplots(figsize=(10, max(4, len(s) * 0.4 + 2)))
    for i, (feat, row) in enumerate(s.iterrows()):
        ax.plot([row['CI_lower'], row['CI_upper']], [i, i],
                color=color, linewidth=2, alpha=0.8)
        ax.scatter(row['HR'], i, color=color, s=60, zorder=3)
        ax.text(-0.01, i, f"{feat}  {row['stars']}",
                va='center', ha='right', fontsize=8,
                transform=ax.get_yaxis_transform())

    ax.axvline(x=1, color='grey', linestyle='--', linewidth=1)
    ax.set_yticks([])
    ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=11)
    ax.set_title(f'Forest Plot — {model_name}  |  C-index={c_idx:.4f}',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.subplots_adjust(left=0.35)
    plt.tight_layout()
    plt.show()

# ── Save ──────────────────────────────────────────────────────────────────────

# all_hr_rows = []
# for model_name, (cph, df_model, features) in fitted_models.items():
#     s = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
#     s.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
#     s['Model']   = model_name
#     s['Feature'] = s.index
#     all_hr_rows.append(s)
# all_hr_df = pd.concat(all_hr_rows).reset_index(drop=True)

# results_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_comparison.csv', index=False)
# lrt_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_lrt.csv', index=False)
# all_hr_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_hazard_ratios.csv', index=False)

print("Done. All results saved.")

In [ ]:
print(f"Complete cases: {len(cox_complete)}")
print(f"Deaths: {int(cox_complete['event'].sum())}")
print(f"Events per variable (X1 strain): {int(cox_complete['event'].sum()) / len(STRAIN_COLS):.1f}")
print(f"Events per variable (X4 VAE):    {int(cox_complete['event'].sum()) / len(VAE_COLS):.1f}")

In [ ]:
# ── Nested Cox — each model on its own maximum sample ─────────────────────────
# LRT is NOT valid across models with different N, so we report
# C-index + Wald test p-values per feature instead of LRT

fitted_models = {}
model_summary = []

for model_name, features in models.items():
    df_model = cox_base[['time', 'event'] + features].dropna()

    try:
        cph = CoxPHFitter(penalizer=0.0)
        cph.fit(df_model, duration_col='time', event_col='event')
        fitted_models[model_name] = (cph, df_model, features)

        model_summary.append({
            'Model':          model_name,
            'N':              len(df_model),
            'N_Events':       int(df_model['event'].sum()),
            'EPV':            round(int(df_model['event'].sum()) / len(features), 1),
            'C_index':        round(cph.concordance_index_, 4),
            'Log_Likelihood': round(cph.log_likelihood_, 4),
            'AIC':            round(cph.AIC_partial_, 4),
            'N_Features':     len(features),
        })

        print(f"{model_name}")
        print(f"  N={len(df_model)} | Events={int(df_model['event'].sum())} | "
              f"EPV={int(df_model['event'].sum())/len(features):.1f} | "
              f"C-index={cph.concordance_index_:.4f}\n")

    except Exception as e:
        print(f"  FAILED — {model_name}: {e}\n")

results_df = pd.DataFrame(model_summary)

# ── LRT only where samples are comparable ─────────────────────────────────────
# X0 vs X4 (VAE): same N=30434 — valid LRT
# X0 vs X3 (LVEF): N=26733 vs 30434 — close enough, note caveat
# X0 vs X1 (Strain): N=25675 vs 30434 — different sample, use Wald instead
# X0 vs X2 (LVMI): N=2568 vs 30434 — very different, use Wald instead

print("=" * 80)
print("LRT — only for models with comparable sample sizes")
print("=" * 80)

# X0 refit on exact same sample as each comparison model for valid LRT
for model_name, features in models.items():
    if model_name == 'X0 — Base (T1 + age + sex)':
        continue

    df_full   = cox_base[['time', 'event'] + BASE + features].dropna()
    df_base   = df_full[['time', 'event'] + BASE]
    df_aug    = df_full[['time', 'event'] + features]

    try:
        # Refit base on same sample as augmented model
        cph_base_matched = CoxPHFitter(penalizer=0.0)
        cph_base_matched.fit(df_base, duration_col='time', event_col='event')

        cph_aug = CoxPHFitter(penalizer=0.0)
        cph_aug.fit(df_aug, duration_col='time', event_col='event')

        n_events  = int(df_full['event'].sum())
        df_diff   = len(features) - len(BASE)
        lr_stat   = 2 * (cph_aug.log_likelihood_ - cph_base_matched.log_likelihood_)
        p_val     = stats.chi2.sf(lr_stat, df_diff)
        ci_imp    = round(cph_aug.concordance_index_ - cph_base_matched.concordance_index_, 4)
        epv       = round(n_events / len(features), 1)
        stars     = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))

        print(f"  {model_name}")
        print(f"    Matched sample: N={len(df_full)} | Events={n_events} | EPV={epv}")
        print(f"    Base C-index={cph_base_matched.concordance_index_:.4f} → "
              f"Augmented C-index={cph_aug.concordance_index_:.4f} (Δ={ci_imp:+.4f})")
        print(f"    LR={lr_stat:.2f} df={df_diff} p={p_val:.3e} {stars}\n")

    except Exception as e:
        print(f"  FAILED — {model_name}: {e}\n")
        
for model_name, features in models.items():
    if model_name == 'X0 — Base (T1 + age + sex)':
        continue

    # extra features only (not BASE — already regressed for age/sex)
    extra_features = [f for f in features if f not in BASE]
    if not extra_features:
        continue

    # matched sample: rows with time, event, BASE, and extra features all present
    df_full = cox_base[['time', 'event'] + BASE + extra_features].dropna()

    try:
        # base model on matched sample
        cph_base_matched = CoxPHFitter(penalizer=0.0)
        cph_base_matched.fit(df_full[['time', 'event'] + BASE],
                             duration_col='time', event_col='event')

        # augmented model: BASE + extra features
        cph_aug = CoxPHFitter(penalizer=0.0)
        cph_aug.fit(df_full[['time', 'event'] + BASE + extra_features],
                    duration_col='time', event_col='event')

        n_events = int(df_full['event'].sum())
        df_diff  = len(extra_features)
        lr_stat  = 2 * (cph_aug.log_likelihood_ - cph_base_matched.log_likelihood_)
        p_val    = stats.chi2.sf(lr_stat, df_diff)
        ci_imp   = round(cph_aug.concordance_index_ - cph_base_matched.concordance_index_, 4)
        epv      = round(n_events / (len(BASE) + len(extra_features)), 1)
        stars    = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))

        print(f"  {model_name}")
        print(f"    Matched sample: N={len(df_full)} | Events={n_events} | EPV={epv}")
        print(f"    Base C-index={cph_base_matched.concordance_index_:.4f} → "
              f"Aug C-index={cph_aug.concordance_index_:.4f} (Δ={ci_imp:+.4f})")
        print(f"    LR={lr_stat:.2f} df={df_diff} p={p_val:.3e} {stars}\n")

    except Exception as e:
        print(f"  FAILED — {model_name}: {e}\n")

# ── C-index summary plot ───────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
colors_list = [model_colors[m] for m in results_df['Model']]
bars = ax.barh(results_df['Model'], results_df['C_index'],
               color=colors_list, alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars, results_df.iterrows()):
    ax.text(bar.get_width() + 0.001,
            bar.get_y() + bar.get_height() / 2,
            f"C={row['C_index']:.4f}  N={row['N']}  events={row['N_Events']}",
            va='center', fontsize=8)

ax.set_xlabel('C-index (concordance)', fontsize=11)
ax.set_title('Nested Cox — C-index by Feature Set\n(each model on its own maximum sample)',
             fontsize=12, fontweight='bold')
ax.set_xlim(0.5, min(results_df['C_index'].max() + 0.1, 1.0))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Forest plots ───────────────────────────────────────────────────────────────

for model_name, (cph, df_model, features) in fitted_models.items():
    s = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
    s.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
    s['stars'] = s['p_value'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
    s = s.sort_values('HR', ascending=True)
    color = model_colors[model_name]
    c_idx = results_df.loc[results_df['Model'] == model_name, 'C_index'].values[0]
    n_ev  = results_df.loc[results_df['Model'] == model_name, 'N_Events'].values[0]

    fig, ax = plt.subplots(figsize=(10, max(4, len(s) * 0.4 + 2)))
    for i, (feat, row) in enumerate(s.iterrows()):
        ax.plot([row['CI_lower'], row['CI_upper']], [i, i],
                color=color, linewidth=2, alpha=0.8)
        ax.scatter(row['HR'], i, color=color, s=60, zorder=3)
        ax.text(-0.01, i, f"{feat}  {row['stars']}",
                va='center', ha='right', fontsize=8,
                transform=ax.get_yaxis_transform())

    ax.axvline(x=1, color='grey', linestyle='--', linewidth=1)
    ax.set_yticks([])
    ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=11)
    ax.set_title(f'Forest Plot — {model_name}\nC-index={c_idx:.4f}  |  Events={n_ev}',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.subplots_adjust(left=0.35)
    plt.tight_layout()
    plt.show()

# ── Save ──────────────────────────────────────────────────────────────────────

# all_hr_rows = []
# for model_name, (cph, df_model, features) in fitted_models.items():
#     s = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
#     s.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
#     s['Model']   = model_name
#     s['Feature'] = s.index
#     all_hr_rows.append(s)
# all_hr_df = pd.concat(all_hr_rows).reset_index(drop=True)

# results_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_comparison.csv', index=False)
# all_hr_df.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update2/nested_cox_hazard_ratios.csv', index=False)
print("Done.")

## Kaplan–Meier Curves: Cardiovascular Mortality · STEMI · Stroke

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ── Base ─────────────────────────────────────────────────────────────────
base = big_pheno_data.drop_duplicates(subset='id', keep='first').copy()
base['time']  = pd.to_numeric(base['mri_to_event'], errors='coerce')
base['event'] = base['death_boolean'].astype(int)
base = base[(base['time'] > 0) & base['time'].notna() & base['sex'].notna()].reset_index(drop=True)
print(f"base: n={len(base)}  deaths={base['event'].sum()}")

# ── Event flags ──────────────────────────────────────────────────────────
base['event_cv']     = ((base['death_boolean'] == True) &
                         base['icd10'].astype(str).str.contains(r'\bI\d{2}', na=False, regex=True)).astype(int)
base['event_mi']     = base['icd10'].astype(str).str.contains(r'\bI21', na=False, regex=True).astype(int)
base['event_stroke'] = pd.to_numeric(base['stroke'], errors='coerce').fillna(0).astype(int)

print(f"CV deaths={base['event_cv'].sum()}  STEMI={base['event_mi'].sum()}  Stroke={base['event_stroke'].sum()}")
print(base['sex'].value_counts())

# ── Plot ─────────────────────────────────────────────────────────────────
def plot_km(ax, df, event_col, title):
    for grp in sorted(df['sex'].unique()):
        mask = df['sex'] == grp
        kmf = KaplanMeierFitter()
        kmf.fit(df.loc[mask, 'time'], df.loc[mask, event_col], label=grp)
        kmf.plot_survival_function(ax=ax, ci_show=True)
    grps = sorted(df['sex'].unique())
    if len(grps) == 2:
        m0, m1 = df['sex'] == grps[0], df['sex'] == grps[1]
        lr = logrank_test(df.loc[m0,'time'], df.loc[m1,'time'],
                          df.loc[m0, event_col], df.loc[m1, event_col])
        ax.set_title(f"{title}\np={lr.p_value:.3e}")
    ax.set_xlabel('Time from MRI (years)')
    ax.set_ylabel('Survival probability')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_km(axes[0], base, 'event_cv',     'CV Mortality')
plot_km(axes[1], base, 'event_mi',     'STEMI / Incident MI')
plot_km(axes[2], base, 'event_stroke', 'Stroke')
plt.tight_layout()
plt.show()

In [ ]:
survival_df = pd.DataFrame(merged_data['id'])
study_end = pd.to_datetime('2022-10-18')
merged_data['date_death_dt'] = pd.to_datetime(merged_data['date_death'])
survival_df['time'] = (
    merged_data['date_death_dt'].fillna(study_end) - 
    merged_data['estimated_mri_date']
).dt.days / 365.25
survival_df['event'] = merged_data['death_boolean'].astype(int)
survival_df['group'] = big_pheno_data['sex']
survival_data = pd.merge(survival_df, unet_t1_regressed, left_on="id", right_on="IID")
survival_data = pd.merge(survival_data, latent_dimensions, left_on="id", right_on="IID")
survival_data = survival_data.drop(["IID_x", "FID_x", "IID_y", "FID_y"], axis=1)  # keep id this time

# Now attach cause-specific events
base_indexed = base.set_index('id')
survival_data['event_cv']     = survival_data['id'].map(base_indexed['event_cv'])
survival_data['event_mi']     = survival_data['id'].map(base_indexed['event_mi'])

print(survival_data[['event_cv','event_mi']].sum())
print(f"n={len(survival_data)}")

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

phenotypes = [c for c in survival_data.columns 
              if c not in ['time', 'event', 'group', 'id', 'event_cv', 'event_mi']]

results_list = []

for event_col, event_label in [('event_cv', 'CV Mortality'),
                                ('event_mi', 'STEMI')]:
    for phenotype in phenotypes:
        df = survival_data[['time', event_col, phenotype]].dropna().copy()
        cutoff = df[phenotype].median()
        df['group'] = np.where(df[phenotype] > cutoff, 'High', 'Low')

        lr = logrank_test(df.loc[df['group']=='Low',  'time'], df.loc[df['group']=='High', 'time'],
                          df.loc[df['group']=='Low',  event_col], df.loc[df['group']=='High', event_col])

        results_list.append({
            'endpoint':  event_label,
            'phenotype': phenotype,
            'p_value':   lr.p_value,
            'test_stat': lr.test_statistic,
            'n':         len(df),
            'n_events':  int(df[event_col].sum()),
            'cutoff':    cutoff,
        })

results_df = pd.DataFrame(results_list).sort_values('p_value')

In [ ]:
sig = results_df[results_df['p_value'] < 0.05 / len(results_df)]
print(f"\n{len(sig)} significant results (p<0.05)")

sig

# Disease Grouping

In [ ]:
valid_images = np.load("./data/shriya/SHMOLLI-output-unet-myocardium-update2/quality_ID_list.csv.npy").tolist()
bad_images = np.load("./data/shriya/SHMOLLI-output-unet-myocardium-update2/bad_ID_list.csv.npy").tolist()

valid_images = [int(x) for x in valid_images]
valid_ids = set(valid_images)

survival_data = survival_data[survival_data["id"].isin(valid_ids)]

In [ ]:
labelled_pheno_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/cleaned_mean_T1_allpheno.csv", sep=',', header=0)

def parse_ids_to_dict(text_content):
    # Initialize the dictionary to store the IDs
    cardiac_ids = {}
    
    # Use a pattern to find all disease categories in the text
    disease_pattern = r"([A-Za-z\s]+) IDs: (.*?)(?:\nNumber|\Z)"
    
    # Find all matches in the text
    disease_matches = re.finditer(disease_pattern, text_content, re.DOTALL)
    
    # Process each disease category found
    for match in disease_matches:
        disease_name = match.group(1).strip()
        ids_str = match.group(2).strip()
        
        # Convert string of IDs to list of integers
        try:
            ids_list = [int(id_str.strip()) for id_str in ids_str.split(',')]
            cardiac_ids[disease_name] = ids_list
        except ValueError as e:
            print(f"Error parsing IDs for {disease_name}: {e}")
    
    return cardiac_ids

with open('./data/shriya/SHMOLLI-output-unet-myocardium/disease_patient_IDs.txt', 'r') as file:
    text_content = file.read()
    cardiac_ids_dict = parse_ids_to_dict(text_content)

In [ ]:
type1_diabetes_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("E10", na=False)]["id"])
type2_diabetes_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("E11", na=False)]["id"])

MI_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I21", na=False)]["id"])
DCM_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I42", na=False)]["id"])

sarcoidosis_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("D86", na=False)]["id"])
chronic_kidney_disease_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("N18", na=False)]["id"])
hypertension_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I10", na=False)]["id"])

In [ ]:
HCM_patients = cardiac_ids_dict["HCM"]
Valvular_patients = cardiac_ids_dict["Valvular"]
Amyloidosis_patients = cardiac_ids_dict["Amyloidosis"]
Ischemic_patients = cardiac_ids_dict["Ischemic"]
NonIschemic_patients = cardiac_ids_dict["NonIschemic"]
Normal_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].isna()]["id"])

In [ ]:
disease_groups = {
    'Type 1 Diabetes': type1_diabetes_patients,
    'Type 2 Diabetes': type2_diabetes_patients,
    'Myocardial Infarction': MI_patients,
    'Dilated Cardiomyopathy': DCM_patients,
    'Sarcoidosis': sarcoidosis_patients,
    'Chronic Kidney Disease': chronic_kidney_disease_patients,
    'Hypertension': hypertension_patients,
    'HCM': HCM_patients,
    'Valvular Disease': Valvular_patients,
    'Amyloidosis': Amyloidosis_patients,
    'Ischemic Heart Disease': Ischemic_patients,
    'Non-Ischemic Heart Disease': NonIschemic_patients,
    'Normal': Normal_patients
}

In [ ]:
from scipy import stats
import numpy as np

def calculate_disease_prevalence(survival_data, phenotype, disease_groups):
    """
    Calculate disease prevalence for each quartile of a phenotype.
    Returns prevalence data and p-values.
    """
    survival_data_copy = survival_data.copy()
    
    # Create quartiles for the phenotype
    survival_data_copy['quartile'] = pd.qcut(
        survival_data_copy[phenotype], 
        q=4, 
        labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
    )
    
    # Initialize results dictionary
    results = {
        'disease': [],
        'quartile': [],
        'prevalence': [],
        'p_value': [],
        'total_patients': []
    }
    
    # For each disease, calculate its prevalence in each quartile
    for disease_name, disease_patients in disease_groups.items():
        # Convert disease patient list to a set for faster lookups
        disease_patients_set = set(disease_patients)
        
        # Get counts for each quartile
        quartile_counts = {}
        quartile_totals = {}
        
        for quartile in ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']:
            quartile_patients = set(survival_data_copy[survival_data_copy['quartile'] == quartile]['id'])
            quartile_disease_patients = quartile_patients.intersection(disease_patients_set)
            
            quartile_counts[quartile] = len(quartile_disease_patients)
            quartile_totals[quartile] = len(quartile_patients)
            
            # Calculate prevalence (percentage)
            prevalence = (len(quartile_disease_patients) / len(quartile_patients)) * 100 if len(quartile_patients) > 0 else 0
            
            results['disease'].append(disease_name)
            results['quartile'].append(quartile)
            results['prevalence'].append(prevalence)
            results['total_patients'].append(len(quartile_patients))
        
        # Calculate p-value using Chi-square test
        # Create contingency table: rows are quartiles, columns are [disease, no_disease]
        contingency_table = np.array([
            [quartile_counts['Q1 (Lowest)'], quartile_totals['Q1 (Lowest)'] - quartile_counts['Q1 (Lowest)']],
            [quartile_counts['Q2'], quartile_totals['Q2'] - quartile_counts['Q2']],
            [quartile_counts['Q3'], quartile_totals['Q3'] - quartile_counts['Q3']],
            [quartile_counts['Q4 (Highest)'], quartile_totals['Q4 (Highest)'] - quartile_counts['Q4 (Highest)']]
        ])
        
        # Only perform the test if we have sufficient data
        if np.all(contingency_table.sum(axis=1) > 0):
            chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
        else:
            p_value = np.nan
        
        # Add p-value to the results (same p-value for all quartiles of this disease)
        for i in range(4):  # 4 quartiles
            results['p_value'].append(p_value)
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

#Function to plot disease prevalence across quartiles

def plot_disease_prevalence(results_df, phenotype):
    """
    Create a grid of histograms showing disease prevalence across quartiles.
    """
    # Get unique diseases
    diseases = results_df['disease'].unique()
    
    # Create figure with subplots
    n_diseases = len(diseases)
    fig = plt.figure(figsize=(20, n_diseases * 3))
    gs = GridSpec(n_diseases, 1)
    
    # Color palette for quartiles
    colors = sns.color_palette("viridis", 4)
    
    for i, disease in enumerate(diseases):
        ax = fig.add_subplot(gs[i])
        
        # Filter data for this disease
        disease_data = results_df[results_df['disease'] == disease]
        
        # Sort by quartile to ensure proper order
        quartile_order = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
        disease_data = disease_data.set_index('quartile').loc[quartile_order].reset_index()
        
        # Plot bars
        bars = ax.bar(disease_data['quartile'], disease_data['prevalence'], color=colors)
        
        # Add p-value annotation
        p_value = disease_data['p_value'].iloc[0]  # Same p-value for all quartiles of this disease
        p_value_text = f"p = {p_value:.4f}" if not np.isnan(p_value) else "p = N/A"
        significance = ""
        if not np.isnan(p_value):
            if p_value < 0.001:
                significance = "***"
            elif p_value < 0.01:
                significance = "**"
            elif p_value < 0.05:
                significance = "*"
        
        ax.text(0.95, 0.95, p_value_text + significance, 
                transform=ax.transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Add percentage and count labels on top of each bar
        for bar, prev, total in zip(bars, disease_data['prevalence'], disease_data['total_patients']):
            count = int(prev * total / 100)
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f"{prev:.1f}%\n({count})",
                    ha='center', va='bottom', fontsize=9)
        
        # Set titles and labels
        ax.set_title(f"{disease} Prevalence by {phenotype} Quartile", fontsize=14)
        ax.set_ylabel('Prevalence (%)', fontsize=12)
        ax.set_ylim(0, max(disease_data['prevalence']) * 1.2)  # Add 20% margin for labels
        
        # Add grid lines for better readability
        ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec

all_results = pd.DataFrame(columns=[
    'disease', 'phenotype', 'quartile', 'prevalence', 'p_value', 
    'total_patients', 'disease_patients'
])

for i in survival_data.columns:
    # Skip non-phenotype columns
    if i in ["time", "event", "group", "id"]:
        continue
    
    phenotype = i
    print(f"Processing phenotype: {phenotype}")
    
    # Calculate disease prevalence
    results_df = calculate_disease_prevalence(survival_data, phenotype, disease_groups)
    
    # Create plots
    plot_disease_prevalence(results_df, phenotype)
    
    # Add phenotype column to results dataframe
    results_df['phenotype'] = phenotype
    
    # Calculate disease patients counts
    results_df['disease_patients'] = results_df.apply(
        lambda row: int(row['prevalence'] * row['total_patients'] / 100), 
        axis=1
    )
    
    # Append to master results dataframe
    all_results = pd.concat([all_results, results_df], ignore_index=True)

In [ ]:
print(f"Analysis complete. Results saved to 'analysis_results/phenotype_disease_prevalence.csv'")
print(f"Total records: {len(all_results)}")
print(f"Unique phenotypes analyzed: {all_results['phenotype'].nunique()}")
print(f"Unique diseases analyzed: {all_results['disease'].nunique()}")

# Create a summary of significant findings (p < 0.05)
significant_findings = all_results.copy()
significant_findings = significant_findings[significant_findings['p_value'] < 0.05/20 * all_results['disease'].nunique()]
significant_findings = significant_findings.sort_values('prevalence')
#significant_findings = significant_findings.drop_duplicates(subset=['p_value'])
significant_findings = significant_findings.sort_values('p_value')
significant_findings = significant_findings.sort_values('prevalence')
significant_findings = significant_findings.sort_values('phenotype')

In [ ]:
significant_findings.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/chi2_disease_prevalence.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
import matplotlib.patches as mpatches

# Calculate -log10(p-value)
significant_findings['neg_log_pval'] = -np.log10(significant_findings['p_value'])

# Prepare the data
# Pivot to get disease as rows and phenotypes as columns with -log10(p) as values
heatmap_data = significant_findings.pivot_table(
    index='disease',
    columns='phenotype',
    values='neg_log_pval',
    aggfunc='max'  # In case there are duplicates, take the max
)

# Optional: Order diseases by your preference
disease_order = [
    'Normal', 'Hypertension', 'Type 2 Diabetes', 'Ischemic Heart Disease',
    'Myocardial Infarction', 'Valvular Disease', 'Type 1 Diabetes',
    'Non-Ischemic Heart Disease', 'Sarcoidosis'
]
# Keep only diseases that exist in the data
disease_order = [d for d in disease_order if d in heatmap_data.index]
heatmap_data = heatmap_data.reindex(disease_order)

# Separate columns into two groups
latent_cols = ['Latent_1', 'Latent_2', 'Latent_3', 'Latent_4', 'Latent_5', 'Latent_6', 'Latent_7',
       'Latent_9', 'Latent_10', 'Latent_11', 'Latent_14', 'Latent_15',]
t1_cols = [col for col in heatmap_data.columns if 'hematocrit' not in col and 'T1' in col]

# Reorder columns: T1 features first, then Latent dimensions
ordered_cols = t1_cols + latent_cols
heatmap_data = heatmap_data[ordered_cols]

# Define color mapping based on -log10(p-values)
# White: < 1.3 (p > 0.05), Light pink: 1.3-2 (p < 0.05), 
# Medium pink: 2-3 (p < 0.01), Red: 3-4 (p < 0.001), Dark red: > 4 (p < 0.0001)
colors = ['#FFFFFF', '#FFB6C1', '#FF69B4', '#DC143C', '#8B0000']
boundaries = [0, 1.3, 2.0, 3.0, 4.0, 10.0]
cmap = LinearSegmentedColormap.from_list('custom', colors, N=256)
norm = BoundaryNorm(boundaries, cmap.N)

# Create figure
fig, ax = plt.subplots(figsize=(16, 8))

# Create heatmap
sns.heatmap(
    heatmap_data,
    cmap=cmap,
    norm=norm,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    linecolor='lightgray',
    cbar=False,
    ax=ax,
    annot_kws={'size': 8}
)

# Customize
ax.set_xlabel('')
ax.set_ylabel('Disease', fontsize=12, fontweight='bold')
ax.set_title('Cardiac Disease Association Analysis: Confounder-Adjusted T1\nMapping versus Latent Dimension',
             fontsize=14, fontweight='bold', pad=20)

# Rotate x-axis labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

# Add column group labels
t1_start = 0
t1_end = len(t1_cols)
latent_start = t1_end
latent_end = len(ordered_cols)

# Add colored bars above to indicate groups
ax.add_patch(mpatches.Rectangle((t1_start, -0.5), t1_end - t1_start, 0.3,
                                 facecolor='#FF6B6B', clip_on=False, 
                                 transform=ax.transData, zorder=10))
ax.add_patch(mpatches.Rectangle((latent_start, -0.5), latent_end - latent_start, 0.3,
                                 facecolor='#9B59B6', clip_on=False,
                                 transform=ax.transData, zorder=10))

# Add group text labels
ax.text((t1_start + t1_end) / 2, -1.0, 'T1 Relaxation Time Scalars',
        ha='center', va='bottom', fontsize=11, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#FF6B6B', alpha=0.8))
ax.text((latent_start + latent_end) / 2, -1.0, 'VAE Latent Dimensions',
        ha='center', va='bottom', fontsize=11, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#9B59B6', alpha=0.8))

# # Create custom legend with -log10(p) values
# legend_elements = [
#     mpatches.Patch(facecolor='#FFFFFF', edgecolor='black', label='Not Significant (-log10(p) < 1.3)'),
#     mpatches.Patch(facecolor='#FFB6C1', label='p < 0.05 (-log10(p): 1.3-2.0)'),
#     mpatches.Patch(facecolor='#FF69B4', label='p < 0.01 (-log10(p): 2.0-3.0)'),
#     mpatches.Patch(facecolor='#DC143C', label='p < 0.001 (-log10(p): 3.0-4.0)'),
#     mpatches.Patch(facecolor='#8B0000', label='p < 0.0001 (-log10(p) > 4.0)')
# ]
# ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.05, 0.5),
#           frameon=True, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

def cohens_d(group1, group2):
    """
    Calculate Cohen's d effect size
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    # Cohen's d
    d = (np.mean(group1) - np.mean(group2)) / pooled_std
    
    return d

def compare_disease_phenotypes(survival_data, disease_groups):
    """
    Compare median phenotype values between disease and non-disease groups
    Calculate Cohen's d effect sizes
    """
    # Get phenotype columns (exclude id, time, event, group)
    phenotype_cols = [col for col in survival_data.columns 
                     if col not in ['id', 'time', 'event', 'group'] and 'regressed_hematocrit' not in col]
    
    results = []
    
    # For each disease
    for disease_name, disease_ids in disease_groups.items():
        print(f"Processing {disease_name}...")
        
        # Create disease vs non-disease groups
        disease_mask = survival_data['id'].isin(disease_ids)
        disease_group = survival_data[disease_mask]
        non_disease_group = survival_data[~disease_mask]
        
        n_disease = len(disease_group)
        n_non_disease = len(non_disease_group)
        
        # For each phenotype
        for phenotype in phenotype_cols:
            # Remove NaN values
            disease_values = disease_group[phenotype].dropna()
            non_disease_values = non_disease_group[phenotype].dropna()
            
            if len(disease_values) > 0 and len(non_disease_values) > 0:
                # Calculate medians
                median_disease = disease_values.median()
                median_non_disease = non_disease_values.median()
                
                # Calculate means for Cohen's d
                mean_disease = disease_values.mean()
                mean_non_disease = non_disease_values.mean()
                
                # Calculate Cohen's d
                effect_size = cohens_d(disease_values.values, non_disease_values.values)
                
                # Mann-Whitney U test (non-parametric test for medians)
                statistic, p_value = stats.mannwhitneyu(
                    disease_values, 
                    non_disease_values, 
                    alternative='two-sided'
                )
                
                # Store results
                results.append({
                    'disease': disease_name,
                    'phenotype': phenotype,
                    'n_disease': n_disease,
                    'n_non_disease': n_non_disease,
                    'median_disease': median_disease,
                    'median_non_disease': median_non_disease,
                    'mean_disease': mean_disease,
                    'mean_non_disease': mean_non_disease,
                    'cohens_d': effect_size,
                    'p_value': p_value,
                    'direction': 'increasing' if median_disease > median_non_disease else 'decreasing'
                })
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    # Add significance flags
    results_df['significant'] = results_df['p_value'] < 0.05
    results_df['neg_log_pval'] = -np.log10(results_df['p_value'])
    
    return results_df

# Run the analysis
disease_phenotype_results = compare_disease_phenotypes(survival_data, disease_groups)

disease_phenotype_results

In [ ]:
disease_phenotype_results[disease_phenotype_results["p_value"] < 0.05/325]